# Historical edge audit: Paper vs independent recorder

## TL;DR

No tradable edge is established. Paper's strongest segment is BTC 5m at Beijing 20:00–24:00, and a five-minute cooldown is directionally positive inside Paper. Both fail the independent touch-recorder check: EV remains negative using observed first-touch asks. The fill-model disagreement is larger than the candidate strategy EV, so measurement reconciliation comes before strategy deployment.

## Context and method

- Unit: one settled Dual-entry profile/round.
- Timezone: Asia/Shanghai (UTC+8).
- Discovery: through 2026-08-12; holdout: after 2026-08-12.
- Cooldown uses only outcomes settled before the next entry.
- Day-block bootstrap preserves within-day dependence.
- First and last Paper calendar days are partial. Strategy-check features were unavailable in the bounded extract.
- Independent validation uses BTC 5m recorder rounds at a 29c ceiling, priced at each side's observed first bestAsk.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
frame = pd.read_csv(ROOT / 'rounds_snapshot.csv')
results = json.loads((ROOT / 'analysis_results.json').read_text())
recorder = json.loads((ROOT / 'recorder_validation.json').read_text())
validation = json.loads((ROOT / 'validation_results.json').read_text())
frame.shape, results['as_of'], validation['status']

((2689, 27), '2026-08-17T06:42:05.100Z', 'pass')

## Data quality

In [2]:
dq = results['data_quality']
pd.Series({
    'round rows': dq['round_rows'],
    'unique keys': dq['unique_round_keys'],
    'duplicate keys': dq['duplicate_round_keys'],
    'fill partition errors': dq['both_single_partition_errors'],
    'strategy feature availability': 'unavailable',
}).to_frame('value')

,value
round rows,2689
unique keys,2689
duplicate keys,0
fill partition errors,0
strategy feature availability,unavailable


## Primary policy comparison

In [3]:
names = {
 'baseline': 'Baseline',
 'cooldown_5m_after_single': '5m cooldown after single',
 'utc8_00_04': 'UTC+8 00–04',
 'btc5m_utc8_00_04': 'BTC 5m, UTC+8 00–04',
 'cooldown_5m_and_utc8_00_04_or_20_24': '5m cooldown + 00–04/20–24',
}
rows=[]
for key,label in names.items():
    item=results['policy_diagnostics'][key]
    m=item['metrics']
    rows.append({
      'policy':label,'rounds':m['n'],'pnl':m['pnl'],'ev_per_round':m['ev'],
      'roi_pct':m['roi_pct'],'positive_days':f"{m['positive_days']}/{m['covered_days']}",
      'daily_std':m['daily_pnl_std'],'max_drawdown':m['daily_max_drawdown'],
      'discovery_ev':item['discovery']['ev'],'holdout_ev':item['holdout']['ev'],
      'bootstrap_low':item['block_bootstrap_ev95']['low'],'bootstrap_high':item['block_bootstrap_ev95']['high'],
    })
policy_table=pd.DataFrame(rows)
display(policy_table)

,policy,rounds,pnl,ev_per_round,roi_pct,positive_days,daily_std,max_drawdown,discovery_ev,holdout_ev,bootstrap_low,bootstrap_high
0,Baseline,2689,-45.355,-0.016867,-0.861,4/8,53.167432,-126.55,-0.026166,-0.007421,-0.124166,0.081506
1,5m cooldown after single,1926,54.050,0.028063,1.341,5/8,35.149085,-56.75,0.030000,0.026078,-0.074882,0.113588
2,UTC+8 00–04,448,15.200,0.033929,1.708,4/7,30.704042,-56.75,0.098698,-0.014648,-0.298103,0.346116
3,"BTC 5m, UTC+8 00–04",336,11.200,0.033333,1.987,4/7,24.392314,-48.20,0.166319,-0.066406,-0.315327,0.371131
4,5m cooldown + 00–04/20–24,655,87.250,0.133206,6.198,7/8,17.294290,-16.70,0.229970,0.033746,-0.002646,0.255339


## Time-block diagnostics

In [4]:
time_rows=[]
for item in results['all_candidates']:
    if item['dimension']=='time_block':
        time_rows.append({
          'time_block':item['value'],'rounds':item['full']['n'],'pnl':item['full']['pnl'],
          'full_ev':item['full']['ev'],'discovery_ev':item['discovery']['ev'],
          'holdout_ev':item['holdout']['ev'],'positive_days':f"{item['full']['positive_days']}/{item['full']['covered_days']}",
        })
display(pd.DataFrame(time_rows))

,time_block,rounds,pnl,full_ev,discovery_ev,holdout_ev,positive_days
0,00-04,448,15.200,0.033929,0.098698,-0.014648,4/7
1,04-08,448,-22.800,-0.050893,-0.230990,0.084180,3/7
2,08-12,458,-12.950,-0.028275,-0.033744,-0.023922,5/8
3,12-16,439,-13.905,-0.031674,-0.072676,0.025683,3/7
4,16-20,448,-54.450,-0.121540,-0.110156,-0.136719,1/7
5,20-24,448,43.550,0.097210,0.170313,-0.000260,5/7


## Cross-source falsification test

In [5]:
paper = validation['btc5m_20_24_paper']['full']
touch = validation['recorder_actual_touch']
agreement = validation['cross_source_agreement']
display(pd.DataFrame([
 {'source':'Paper, BTC5m 20–24','rounds':paper['n'],'ev':paper['ev'],
  'ci_low':validation['btc5m_20_24_paper']['block_bootstrap_ev95']['low'],
  'ci_high':validation['btc5m_20_24_paper']['block_bootstrap_ev95']['high']},
 {'source':'Recorder actual touch, all history','rounds':touch['btc5m_20_24_all_history']['rounds'],
  'ev':touch['btc5m_20_24_all_history']['ev_per_share'],
  'ci_low':touch['btc5m_20_24_all_history']['block_bootstrap_ev95']['low'],
  'ci_high':touch['btc5m_20_24_all_history']['block_bootstrap_ev95']['high']},
 {'source':'Recorder cooldown, all history','rounds':touch['cooldown_all_history']['rounds'],
  'ev':touch['cooldown_all_history']['ev_per_share'],
  'ci_low':touch['cooldown_all_history']['block_bootstrap_ev95']['low'],
  'ci_high':touch['cooldown_all_history']['block_bootstrap_ev95']['high']},
]))
display(pd.Series(agreement, name='value').drop('contingency').to_frame())

,source,rounds,ev,ci_low,ci_high
0,"Paper, BTC5m 20–24",336,0.149554,0.027679,0.250446
1,"Recorder actual touch, all history",880,-0.047469,-0.067053,-0.027784
2,"Recorder cooldown, all history",3460,-0.046284,-0.059279,-0.033374


,value
matched_btc5m_rounds,2017
target_rounds,336
paper_paired,128
recorder_paired,45
paired_in_both,44
paper_side_fills,464
paper_only_side_fills,163
paper_only_side_fill_rate,0.351293


## Takeaways

1. Paper baseline is negative and its day-block 95% interval crosses zero.
2. BTC 5m at Beijing 20:00–24:00 is the strongest Paper hypothesis, but independent recorder EV is negative across the longer history.
3. Five-minute cooldown improves Paper and reduces recorder losses, but recorder EV remains significantly below zero. It is a risk-control hypothesis, not alpha.
4. In the target segment, 35.1% of Paper side fills are not corroborated by the recorder; only 44 of 128 Paper paired rounds are paired in both sources.
5. Avoiding 16:00–20:00 is the cleanest pre-registered avoidance arm, but no production strategy change is justified until fill semantics are reconciled.